# The Inverse Folding Problem: From Structure to Sequence

This notebook explores the inverse folding problem and introduces the key concepts behind ProteinMPNN for protein sequence design.

## Learning Objectives

1. Understand forward vs inverse folding
2. Learn how to encode protein structure as a graph
3. Implement message passing on protein structures
4. Build a simple autoregressive decoder for sequence generation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Amino acid vocabulary
AA_VOCAB = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_VOCAB)}
IDX_TO_AA = {i: aa for i, aa in enumerate(AA_VOCAB)}

print(f"Amino acid vocabulary size: {len(AA_VOCAB)}")
print(f"Vocabulary: {AA_VOCAB}")

## 1. Forward vs Inverse Folding

**Forward folding** (Structure Prediction):
- Input: Amino acid sequence
- Output: 3D structure
- Example: AlphaFold2

**Inverse folding** (Sequence Design):
- Input: 3D structure (backbone)
- Output: Amino acid sequence(s)
- Example: ProteinMPNN

The inverse folding problem is **one-to-many**: a single structure can be stabilized by many different sequences.

In [ ]:
# Illustration: Multiple sequences can fold to similar structures

# Example: Cytochrome c from different species
# These sequences have low identity but similar structure
example_sequences = {
    'Human': 'MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE',
    'Horse': 'MGDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFTYTDANKNKGITWKEETLMEYLENPKKYIPGTKMIFAGIKKKTEREDLIAYLKKATNE',
    'Tuna':  'MGDVAKGKKTFVQKCAQCHTVENGGKHKVGPNLWGLFGRKTGQAEGYSYTDANKSKGIVWNNDTLMEYLENPKKYIPGTKMIFAGIKKKGERQDLIAYLKQATAK'
}

# Calculate sequence identity
def sequence_identity(seq1, seq2):
    """Calculate percentage of identical positions."""
    min_len = min(len(seq1), len(seq2))
    matches = sum(1 for a, b in zip(seq1[:min_len], seq2[:min_len]) if a == b)
    return matches / min_len * 100

print("Sequence Identity Between Cytochrome c Orthologs:")
print(f"Human vs Horse: {sequence_identity(example_sequences['Human'], example_sequences['Horse']):.1f}%")
print(f"Human vs Tuna:  {sequence_identity(example_sequences['Human'], example_sequences['Tuna']):.1f}%")
print(f"Horse vs Tuna:  {sequence_identity(example_sequences['Horse'], example_sequences['Tuna']):.1f}%")

print("\nDespite ~60-85% identity, all fold into nearly identical structures!")

## 2. Representing Protein Structure

For inverse folding, we need to represent the protein backbone structure.

Each residue has 4 backbone atoms:
- **N**: Nitrogen (amino group)
- **CA**: Alpha carbon
- **C**: Carbonyl carbon
- **O**: Carbonyl oxygen

In [ ]:
def generate_ideal_helix(n_residues):
    """
    Generate ideal alpha helix backbone coordinates.
    
    Alpha helix parameters:
    - Rise per residue: 1.5 Angstroms
    - Radius: 2.3 Angstroms
    - Residues per turn: 3.6
    """
    coords = {'N': [], 'CA': [], 'C': [], 'O': []}
    
    rise_per_residue = 1.5
    radius = 2.3
    residues_per_turn = 3.6
    
    for i in range(n_residues):
        theta = 2 * np.pi * i / residues_per_turn
        z = rise_per_residue * i
        
        # CA position
        ca_x = radius * np.cos(theta)
        ca_y = radius * np.sin(theta)
        ca_z = z
        
        # N is before CA along the helix
        n_theta = theta - 0.3
        n_x = radius * np.cos(n_theta)
        n_y = radius * np.sin(n_theta)
        n_z = z - 0.5
        
        # C is after CA
        c_theta = theta + 0.3
        c_x = radius * np.cos(c_theta)
        c_y = radius * np.sin(c_theta)
        c_z = z + 0.5
        
        # O is off the C
        o_x = c_x + 1.2 * np.cos(theta + np.pi/2)
        o_y = c_y + 1.2 * np.sin(theta + np.pi/2)
        o_z = c_z
        
        coords['N'].append([n_x, n_y, n_z])
        coords['CA'].append([ca_x, ca_y, ca_z])
        coords['C'].append([c_x, c_y, c_z])
        coords['O'].append([o_x, o_y, o_z])
    
    return {k: torch.tensor(v, dtype=torch.float32) for k, v in coords.items()}

# Generate a 20-residue helix
helix_coords = generate_ideal_helix(20)

print("Backbone coordinates shape:")
for atom, coords in helix_coords.items():
    print(f"  {atom}: {coords.shape}")

In [ ]:
# Visualize the helix backbone
fig = plt.figure(figsize=(12, 5))

# 3D view
ax1 = fig.add_subplot(121, projection='3d')

# Plot CA trace
ca = helix_coords['CA'].numpy()
ax1.plot(ca[:, 0], ca[:, 1], ca[:, 2], 'b-', linewidth=2, label='CA trace')
ax1.scatter(ca[:, 0], ca[:, 1], ca[:, 2], c='blue', s=50)

# Plot N atoms
n = helix_coords['N'].numpy()
ax1.scatter(n[:, 0], n[:, 1], n[:, 2], c='green', s=30, label='N')

# Plot C atoms
c = helix_coords['C'].numpy()
ax1.scatter(c[:, 0], c[:, 1], c[:, 2], c='red', s=30, label='C')

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.legend()
ax1.set_title('Alpha Helix Backbone')

# Top-down view (X-Y plane)
ax2 = fig.add_subplot(122)
ax2.plot(ca[:, 0], ca[:, 1], 'b-', linewidth=2)
ax2.scatter(ca[:, 0], ca[:, 1], c=np.arange(len(ca)), cmap='viridis', s=100)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('Top-Down View (colored by residue index)')
ax2.set_aspect('equal')
plt.colorbar(ax2.collections[0], ax=ax2, label='Residue Index')

plt.tight_layout()
plt.show()

## 3. Building a Graph from Structure

ProteinMPNN represents proteins as graphs:
- **Nodes**: Residues (with their backbone coordinates)
- **Edges**: Connect spatially close residues (k-nearest neighbors)

In [ ]:
def build_knn_graph(ca_coords, k=10):
    """
    Build k-nearest neighbor graph from CA coordinates.
    
    Args:
        ca_coords: [L, 3] CA positions
        k: number of neighbors
    
    Returns:
        edge_index: [2, E] edges
        edge_dist: [E] distances
    """
    L = ca_coords.shape[0]
    
    # Compute pairwise distances
    diff = ca_coords.unsqueeze(0) - ca_coords.unsqueeze(1)  # [L, L, 3]
    dist = diff.norm(dim=-1)  # [L, L]
    
    # Set diagonal to infinity (no self-loops)
    dist.fill_diagonal_(float('inf'))
    
    # Get k nearest neighbors
    _, indices = dist.topk(k, dim=-1, largest=False)  # [L, k]
    
    # Create edge index
    src = torch.arange(L).unsqueeze(-1).expand(-1, k).reshape(-1)
    dst = indices.reshape(-1)
    edge_index = torch.stack([src, dst])
    
    # Get distances
    edge_dist = dist[src, dst]
    
    return edge_index, edge_dist

# Build graph for our helix
k = 10
edge_index, edge_dist = build_knn_graph(helix_coords['CA'], k=k)

print(f"Number of residues: {helix_coords['CA'].shape[0]}")
print(f"Number of edges: {edge_index.shape[1]}")
print(f"Edges per residue: {edge_index.shape[1] / helix_coords['CA'].shape[0]}")
print(f"\nDistance statistics:")
print(f"  Min: {edge_dist.min():.2f} A")
print(f"  Max: {edge_dist.max():.2f} A")
print(f"  Mean: {edge_dist.mean():.2f} A")

In [ ]:
# Visualize the graph
fig = plt.figure(figsize=(14, 5))

# 3D graph view
ax1 = fig.add_subplot(121, projection='3d')

ca = helix_coords['CA'].numpy()

# Draw edges
for i in range(edge_index.shape[1]):
    src, dst = edge_index[:, i]
    ax1.plot([ca[src, 0], ca[dst, 0]],
             [ca[src, 1], ca[dst, 1]],
             [ca[src, 2], ca[dst, 2]],
             'gray', alpha=0.3, linewidth=0.5)

# Draw nodes
ax1.scatter(ca[:, 0], ca[:, 1], ca[:, 2], c='blue', s=100, zorder=5)

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title(f'{k}-Nearest Neighbor Graph')

# Adjacency matrix
ax2 = fig.add_subplot(122)

L = ca.shape[0]
adj_matrix = torch.zeros(L, L)
adj_matrix[edge_index[0], edge_index[1]] = 1

ax2.imshow(adj_matrix.numpy(), cmap='Blues')
ax2.set_xlabel('Residue j')
ax2.set_ylabel('Residue i')
ax2.set_title('Adjacency Matrix')

plt.tight_layout()
plt.show()

## 4. Edge Features

ProteinMPNN uses rich edge features that capture:
- Distance between residues
- Relative orientation
- Sequence separation

In [ ]:
def rbf_encode(distances, num_rbf=16, max_dist=20.0):
    """Radial basis function encoding of distances."""
    centers = torch.linspace(0, max_dist, num_rbf)
    gamma = num_rbf / max_dist
    return torch.exp(-gamma * (distances.unsqueeze(-1) - centers) ** 2)

def compute_local_frame(N, CA, C):
    """Compute local coordinate frame from backbone atoms."""
    # X-axis: CA -> C
    x = F.normalize(C - CA, dim=-1)
    
    # Vector in plane: CA -> N
    v = N - CA
    
    # Z-axis: perpendicular to plane
    z = F.normalize(torch.cross(x, v, dim=-1), dim=-1)
    
    # Y-axis: complete right-handed frame
    y = torch.cross(z, x, dim=-1)
    
    # Stack to form rotation matrix
    R = torch.stack([x, y, z], dim=-1)  # [L, 3, 3]
    
    return R

def compute_edge_features(coords, edge_index, num_rbf=16):
    """
    Compute edge features for ProteinMPNN-style model.
    
    Features:
    - RBF-encoded distance
    - Direction vector in local frame
    - Sequence separation encoding
    """
    src, dst = edge_index
    
    # 1. Distance features
    diff = coords['CA'][dst] - coords['CA'][src]  # [E, 3]
    dist = diff.norm(dim=-1)  # [E]
    rbf_dist = rbf_encode(dist, num_rbf)  # [E, num_rbf]
    
    # 2. Local frames
    frames = compute_local_frame(coords['N'], coords['CA'], coords['C'])
    
    # Direction in source frame
    diff_local = torch.einsum('eij,ej->ei', 
                               frames[src].transpose(-1, -2), 
                               diff)
    direction = diff_local / (dist.unsqueeze(-1) + 1e-8)  # [E, 3]
    
    # 3. Sequence separation
    seq_sep = (dst - src).float()  # [E]
    seq_sep_enc = torch.stack([
        torch.sin(seq_sep * np.pi / 10),
        torch.cos(seq_sep * np.pi / 10),
        torch.sin(seq_sep * np.pi / 5),
        torch.cos(seq_sep * np.pi / 5)
    ], dim=-1)  # [E, 4]
    
    # Concatenate all features
    edge_features = torch.cat([rbf_dist, direction, seq_sep_enc], dim=-1)
    
    return edge_features

# Compute edge features
edge_features = compute_edge_features(helix_coords, edge_index)
print(f"Edge features shape: {edge_features.shape}")
print(f"Feature breakdown: {16} (RBF) + {3} (direction) + {4} (seq sep) = {edge_features.shape[-1]}")

In [ ]:
# Visualize edge features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RBF distance encoding
rbf_feats = edge_features[:, :16].numpy()
axes[0].imshow(rbf_feats[:50].T, aspect='auto', cmap='Blues')
axes[0].set_xlabel('Edge Index')
axes[0].set_ylabel('RBF Bin')
axes[0].set_title('RBF Distance Encoding')

# Direction vectors
dir_feats = edge_features[:, 16:19].numpy()
axes[1].scatter(dir_feats[:, 0], dir_feats[:, 1], c=dir_feats[:, 2], 
                cmap='RdYlBu', alpha=0.6, s=10)
axes[1].set_xlabel('X component')
axes[1].set_ylabel('Y component')
axes[1].set_title('Direction Vectors (color=Z)')
axes[1].set_xlim([-1.1, 1.1])
axes[1].set_ylim([-1.1, 1.1])

# Sequence separation
seq_sep = (edge_index[1] - edge_index[0]).numpy()
axes[2].hist(seq_sep, bins=range(-20, 21), edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Sequence Separation')
axes[2].set_ylabel('Count')
axes[2].set_title('Sequence Separation Distribution')

plt.tight_layout()
plt.show()

## 5. Message Passing on the Graph

ProteinMPNN uses message passing to aggregate structural information from neighbors.

In [ ]:
class MessagePassingLayer(nn.Module):
    """Simple message passing layer."""
    
    def __init__(self, node_dim, edge_dim, hidden_dim=64):
        super().__init__()
        
        self.node_dim = node_dim
        
        # Message function
        self.message_mlp = nn.Sequential(
            nn.Linear(node_dim * 2 + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim)
        )
        
        # Update function
        self.update_mlp = nn.Sequential(
            nn.Linear(node_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim)
        )
        
        self.norm = nn.LayerNorm(node_dim)
        
    def forward(self, x, edge_index, edge_attr):
        """
        Args:
            x: [L, node_dim] node features
            edge_index: [2, E] edges
            edge_attr: [E, edge_dim] edge features
        """
        src, dst = edge_index
        L = x.shape[0]
        
        # Compute messages
        msg_input = torch.cat([x[src], x[dst], edge_attr], dim=-1)
        messages = self.message_mlp(msg_input)  # [E, node_dim]
        
        # Aggregate messages
        aggregated = torch.zeros(L, self.node_dim, device=x.device)
        aggregated.scatter_add_(0, dst.unsqueeze(-1).expand(-1, self.node_dim), messages)
        
        # Update nodes
        update_input = torch.cat([x, aggregated], dim=-1)
        x_new = x + self.update_mlp(update_input)
        x_new = self.norm(x_new)
        
        return x_new

class StructureEncoder(nn.Module):
    """Encode structure using message passing."""
    
    def __init__(self, node_dim=64, edge_dim=23, n_layers=3):
        super().__init__()
        
        # Initial node embedding (from node features)
        self.node_embed = nn.Linear(10, node_dim)  # placeholder
        
        # Edge embedding
        self.edge_embed = nn.Linear(edge_dim, node_dim)
        
        # Message passing layers
        self.layers = nn.ModuleList([
            MessagePassingLayer(node_dim, node_dim) 
            for _ in range(n_layers)
        ])
        
    def forward(self, node_features, edge_index, edge_features):
        # Embed
        x = self.node_embed(node_features)
        e = self.edge_embed(edge_features)
        
        # Message passing
        for layer in self.layers:
            x = layer(x, edge_index, e)
        
        return x

# Test encoder
node_dim = 64
L = helix_coords['CA'].shape[0]

# Placeholder node features
node_features = torch.randn(L, 10)

encoder = StructureEncoder(node_dim=node_dim, edge_dim=edge_features.shape[-1])
encoded = encoder(node_features, edge_index, edge_features)

print(f"Input node features: {node_features.shape}")
print(f"Encoded node features: {encoded.shape}")

In [ ]:
# Visualize encoded features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PCA of encoded features
from sklearn.decomposition import PCA

encoded_np = encoded.detach().numpy()
pca = PCA(n_components=2)
encoded_2d = pca.fit_transform(encoded_np)

axes[0].scatter(encoded_2d[:, 0], encoded_2d[:, 1], 
                c=np.arange(L), cmap='viridis', s=100)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA of Encoded Features')
plt.colorbar(axes[0].collections[0], ax=axes[0], label='Residue Index')

# Heatmap of encoded features
im = axes[1].imshow(encoded_np.T, aspect='auto', cmap='RdBu_r')
axes[1].set_xlabel('Residue Index')
axes[1].set_ylabel('Feature Dimension')
axes[1].set_title('Encoded Feature Heatmap')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Autoregressive Sequence Decoder

ProteinMPNN generates sequences one amino acid at a time, conditioning on:
1. The structure (always available)
2. Previously generated amino acids

In [ ]:
class SimpleDecoder(nn.Module):
    """Simple autoregressive decoder."""
    
    def __init__(self, node_dim=64, n_amino_acids=20, hidden_dim=128):
        super().__init__()
        
        self.n_amino_acids = n_amino_acids
        
        # Amino acid embedding
        self.aa_embed = nn.Embedding(n_amino_acids + 1, node_dim)  # +1 for mask
        
        # Combine structure and sequence
        self.combine = nn.Linear(node_dim * 2, hidden_dim)
        
        # Transformer layers
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=4,
                dim_feedforward=hidden_dim * 4,
                batch_first=True
            ),
            num_layers=2
        )
        
        # Output head
        self.output = nn.Linear(hidden_dim, n_amino_acids)
        
    def forward(self, structure_encoding, sequence, mask=None):
        """
        Args:
            structure_encoding: [L, node_dim]
            sequence: [L] amino acid indices (0-19 or 20 for mask)
            mask: [L, L] causal attention mask
        """
        # Embed sequence
        seq_emb = self.aa_embed(sequence)  # [L, node_dim]
        
        # Combine with structure
        combined = self.combine(torch.cat([structure_encoding, seq_emb], dim=-1))
        
        # Transformer (add batch dimension)
        combined = combined.unsqueeze(0)  # [1, L, hidden_dim]
        if mask is not None:
            transformed = self.transformer(combined, mask=mask)
        else:
            transformed = self.transformer(combined)
        transformed = transformed.squeeze(0)  # [L, hidden_dim]
        
        # Output logits
        logits = self.output(transformed)  # [L, n_amino_acids]
        
        return logits

# Test decoder
decoder = SimpleDecoder(node_dim=node_dim, n_amino_acids=20)

# Create test input with masked sequence
MASK_TOKEN = 20
test_sequence = torch.full((L,), MASK_TOKEN, dtype=torch.long)

logits = decoder(encoded, test_sequence)
print(f"Input sequence: {test_sequence.shape}")
print(f"Output logits: {logits.shape}")

In [ ]:
def sample_sequence(decoder, structure_encoding, temperature=1.0):
    """
    Sample a sequence autoregressively.
    
    Args:
        decoder: decoder model
        structure_encoding: [L, node_dim]
        temperature: sampling temperature
    """
    L = structure_encoding.shape[0]
    MASK_TOKEN = 20
    
    # Initialize with all mask tokens
    sequence = torch.full((L,), MASK_TOKEN, dtype=torch.long)
    
    # Random decoding order
    decoding_order = torch.randperm(L)
    
    # Create causal mask
    order_idx = torch.zeros(L, dtype=torch.long)
    order_idx[decoding_order] = torch.arange(L)
    
    log_probs = []
    
    for step in range(L):
        pos = decoding_order[step].item()
        
        # Create mask (can only see previously decoded positions)
        mask = order_idx.unsqueeze(0) >= order_idx[pos]
        mask = mask.float() * -1e9
        
        # Forward pass
        with torch.no_grad():
            logits = decoder(structure_encoding, sequence, mask.unsqueeze(0))
        
        # Sample at current position
        probs = F.softmax(logits[pos] / temperature, dim=-1)
        aa = torch.multinomial(probs, 1).item()
        
        sequence[pos] = aa
        log_probs.append(torch.log(probs[aa]).item())
    
    return sequence, decoding_order, log_probs

# Sample a sequence
sampled_seq, order, log_probs = sample_sequence(decoder, encoded, temperature=1.0)

# Convert to amino acid string
aa_string = ''.join([IDX_TO_AA[idx.item()] for idx in sampled_seq])

print(f"Sampled sequence: {aa_string}")
print(f"Length: {len(aa_string)}")
print(f"Total log probability: {sum(log_probs):.2f}")

In [ ]:
# Sample multiple sequences and analyze diversity
n_samples = 10
sequences = []

for temp in [0.1, 0.5, 1.0, 2.0]:
    temp_seqs = []
    for _ in range(n_samples):
        seq, _, _ = sample_sequence(decoder, encoded, temperature=temp)
        temp_seqs.append(''.join([IDX_TO_AA[idx.item()] for idx in seq]))
    sequences.append((temp, temp_seqs))

# Analyze diversity
print("Effect of Temperature on Sequence Diversity:")
print("=" * 60)

for temp, seqs in sequences:
    # Count unique sequences
    unique = len(set(seqs))
    
    # Calculate pairwise identity
    identities = []
    for i in range(len(seqs)):
        for j in range(i+1, len(seqs)):
            identities.append(sequence_identity(seqs[i], seqs[j]))
    
    print(f"\nTemperature = {temp}")
    print(f"  Unique sequences: {unique}/{n_samples}")
    print(f"  Mean pairwise identity: {np.mean(identities):.1f}%")
    print(f"  Example: {seqs[0]}")

## 7. Visualize Amino Acid Predictions

Let's visualize what the model predicts at each position.

In [ ]:
# Get probability distributions for all positions
with torch.no_grad():
    logits = decoder(encoded, torch.full((L,), 20, dtype=torch.long))
    probs = F.softmax(logits, dim=-1)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Probability heatmap
im = axes[0].imshow(probs.numpy().T, aspect='auto', cmap='Blues')
axes[0].set_xlabel('Residue Position')
axes[0].set_ylabel('Amino Acid')
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(list(AA_VOCAB))
axes[0].set_title('Amino Acid Probability at Each Position')
plt.colorbar(im, ax=axes[0], label='Probability')

# Entropy at each position
entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1)
axes[1].bar(range(L), entropy.numpy())
axes[1].set_xlabel('Residue Position')
axes[1].set_ylabel('Entropy (nats)')
axes[1].set_title('Prediction Uncertainty (Entropy) at Each Position')
axes[1].axhline(y=np.log(20), color='r', linestyle='--', label='Max entropy (uniform)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Key Takeaways

### What We Learned

1. **Inverse folding** is the problem of designing sequences that fold into a target structure.

2. **Protein structures can be represented as graphs** with residues as nodes and spatial neighbors as edges.

3. **Rich edge features** capture distance, orientation, and sequence separation.

4. **Message passing** aggregates structural information from neighbors.

5. **Autoregressive decoding** generates sequences one amino acid at a time.

6. **Temperature** controls the diversity of generated sequences.

### ProteinMPNN vs Our Simple Model

| Feature | Our Model | ProteinMPNN |
|---------|-----------|-------------|
| Encoder layers | 3 | 3 |
| Decoder layers | 2 | 3 |
| Hidden dim | 64-128 | 128 |
| Edge features | 23 | ~128 |
| Training data | None | 16k+ structures |
| Recovery rate | Random | ~50-60% |

## 9. Exercises

1. **Add more edge features**: Include dihedral angles, hydrogen bond patterns, or secondary structure.

2. **Implement fixed positions**: Modify sampling to fix certain positions (e.g., catalytic residues).

3. **Beam search**: Implement beam search instead of sampling for sequence generation.

4. **Evaluate designs**: Use ESMFold to predict structures of designed sequences and compute TM-scores.

In [ ]:
# Exercise: Implement fixed position sampling

def sample_with_fixed_positions(decoder, structure_encoding, fixed_positions, fixed_aas, temperature=1.0):
    """
    Sample sequence with some positions fixed.
    
    Args:
        fixed_positions: list of position indices to fix
        fixed_aas: list of amino acid indices for fixed positions
    """
    L = structure_encoding.shape[0]
    MASK_TOKEN = 20
    
    # Initialize sequence
    sequence = torch.full((L,), MASK_TOKEN, dtype=torch.long)
    
    # Set fixed positions
    for pos, aa in zip(fixed_positions, fixed_aas):
        sequence[pos] = aa
    
    # Create decoding order (fixed positions first, then random for rest)
    free_positions = [i for i in range(L) if i not in fixed_positions]
    np.random.shuffle(free_positions)
    decoding_order = fixed_positions + free_positions
    
    # Sample only free positions
    for step in range(len(fixed_positions), L):
        pos = decoding_order[step]
        
        with torch.no_grad():
            logits = decoder(structure_encoding, sequence)
        
        probs = F.softmax(logits[pos] / temperature, dim=-1)
        aa = torch.multinomial(probs, 1).item()
        sequence[pos] = aa
    
    return sequence

# Test: Fix positions 5 and 10 as Alanine (A)
fixed_seq = sample_with_fixed_positions(
    decoder, encoded,
    fixed_positions=[5, 10],
    fixed_aas=[AA_TO_IDX['A'], AA_TO_IDX['A']]
)

aa_string = ''.join([IDX_TO_AA[idx.item()] for idx in fixed_seq])
print(f"Sequence with fixed positions: {aa_string}")
print(f"Position 5: {aa_string[5]} (should be A)")
print(f"Position 10: {aa_string[10]} (should be A)")